# Notebook 05 - Baseline Model Training

## Goal

Train a proof-of-concept classifier to predict domain class:

- benign
- phishing
- malware
- spam

Input: `auspex_features_v1.csv`  
Output: model evaluation and saved model for future inference

We'll test both Logistic Regression and Random Forest as candidates.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

In [4]:
features = pd.read_csv("../data/processed/auspex_features_v1.csv")
print(features.shape)
features.head()

(1024538, 11)


,domain,label,length,digit_count,digit_ratio,dot_count,tld,starts_with_digit,contains_www,entropy,hyphen_count
0,cypress.com,benign,11,0,0.000000,1,com,0,0,3.095795,0
1,boy.jp,benign,6,0,0.000000,1,jp,0,0,2.584963,0
2,nnm.ru,benign,6,0,0.000000,1,ru,0,0,2.251629,0
3,1stwebdesigner.com,benign,18,1,0.055556,1,com,1,0,3.794653,0
4,ordasoft.com,benign,12,0,0.000000,1,com,0,0,3.188722,0


In [5]:
# Target
y = features["label"]

# Numeric features
numeric_features = ["length", "digit_count", "digit_ratio", "dot_count",
                    "hyphen_count", "starts_with_digit", "contains_www", "entropy"]

# Categorical features
categorical_features = ["tld"]

# Input
X = features[numeric_features + categorical_features]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (819630, 9), Test: (204908, 9)


In [7]:
# Scaling numeric features
numeric_transformer = StandardScaler()

# One-hot encode TLD
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
import sklearn
print(sklearn.__version__)

1.8.0


In [11]:
print(LogisticRegression)

<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [13]:
logreg = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=500,
        random_state=42
    ))
])

logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

print("Logistic Regression Results")
print("----------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_logreg))
print(classification_report(y_test, y_pred_logreg))
print(confusion_matrix(y_test, y_pred_logreg))

c:\Users\pipza\OneDrive\Desktop\Project-Auspex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Results
----------------------------
Accuracy: 0.7332851816424932
              precision    recall  f1-score   support

      benign       0.98      0.75      0.85    197660
     malware       0.10      0.41      0.16      5341
    phishing       0.13      0.39      0.19      1741
        spam       0.00      0.61      0.01       166

    accuracy                           0.73    204908
   macro avg       0.30      0.54      0.30    204908
weighted avg       0.95      0.73      0.82    204908

[[147292  19620   4168  26580]
 [  1838   2178    568    757]
 [   462    447    684    148]
 [    45     14      5    102]]


In [16]:
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results")
print("--------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Results
--------------------
Accuracy: 0.8006275987272337
              precision    recall  f1-score   support

      benign       0.98      0.81      0.89    197660
     malware       0.11      0.38      0.17      5341
    phishing       0.11      0.47      0.18      1741
        spam       0.01      0.57      0.01       166

    accuracy                           0.80    204908
   macro avg       0.30      0.56      0.31    204908
weighted avg       0.95      0.80      0.87    204908

[[161085  16223   5862  14490]
 [  2128   2051    741    421]
 [   532    285    824    100]
 [    51     18      2     95]]


In [15]:
import os

print(os.cpu_count())

16


In [ ]:
# Compare accuracy and macro F1
acc_logreg = accuracy_score(y_test, y_pred_logreg)
acc_rf = accuracy_score(y_test, y_pred_rf)

if acc_rf >= acc_logreg:
    best_model = rf
    winner_name = "RandomForest"
else:
    best_model = logreg
    winner_name = "LogisticRegression"

# Save model
joblib.dump(best_model, f"../data/processed/{winner_name}_model_v1.pkl")
print(f"Saved {winner_name} model")

In [ ]:
# Example: compute risk score for test set
probs = best_model.predict_proba(X_test)
class_idx = list(best_model.classes_).index("benign")
risk_score = 100 * (1 - probs[:, class_idx])

risk_df = pd.DataFrame({
    "domain": X_test["domain"],
    "label": y_test,
    "predicted": best_model.predict(X_test),
    "risk_score": risk_score
})

risk_df.head(10)

## Notes

- Baseline models trained using existing lexical features.
- Logistic Regression and Random Forest compared.
- Class imbalance handled with class_weight="balanced".
- TLD one-hot encoded; numeric features scaled.
- Risk score defined as 100 * (1 - P(benign)).
- Next steps: integrate model into a notebook prediction pipeline and then move reusable functions to `src/`.